In [1]:
import pandas as pd
import numpy as np
import os
import sys
import gzip
from tqdm.notebook import tqdm
import gc
import pysam
import re
pd.set_option('display.max_columns', None)
import seaborn as sns
import matplotlib.pyplot as plt
import collections

1) Make dict of all genes and their exon coords in CHM13 space
2) classify reads by intersecting their chm13 alignments with male gene exons
3) output read sequence fasta files for each gene "family" e.g., TSPY, RBMY, DAZ

# Find CHM13 Annotation of Male Genes

In [2]:
chm13annotation = pd.read_csv("/project/mkonkel/tangeno/users/giannim/referenceGenomes/chm13_ncbi/data/GCF_009914755.1/genomic_readable.gtf", sep="\t", header=None)
chm13annotation

,0,1,2,3,4,5,6,7,8
0,NC_060925.1,BestRefSeq,gene,7506.0,138480.0,.,-,.,"gene_id ""LOC127239154""; transcript_id """"; db_x..."
1,NC_060925.1,BestRefSeq,transcript,7506.0,138480.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820..."
2,NC_060925.1,BestRefSeq,exon,138321.0,138480.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820..."
3,NC_060925.1,BestRefSeq,exon,129906.0,130010.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820..."
4,NC_060925.1,BestRefSeq,exon,109651.0,109660.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820..."
...,...,...,...,...,...,...,...,...,...
4298395,NC_060948.1,BestRefSeq,transcript,62449384.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561..."
4298396,NC_060948.1,BestRefSeq,exon,62451557.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561..."
4298397,NC_060948.1,BestRefSeq,exon,62451063.0,62451171.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561..."
4298398,NC_060948.1,BestRefSeq,exon,62449384.0,62450563.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561..."


In [3]:
import re
chm13annotation = chm13annotation.rename(columns={0:"contig", 2:"type", 3:"start", 4:"stop", 6:"strand", 8:"info"})
# Define the fields you want to extract
fields = [
    "gene_id", "transcript_id", "db_xref", "description",
    "gbkey", "gene", "gene_biotype", "partial"
]

# Extract each field into its own column
for field in fields:
    chm13annotation[field] = chm13annotation["info"].str.extract(f'{field} "([^"]*)"')
chm13annotation

,contig,1,type,start,stop,5,strand,7,info,gene_id,transcript_id,db_xref,description,gbkey,gene,gene_biotype,partial
0,NC_060925.1,BestRefSeq,gene,7506.0,138480.0,.,-,.,"gene_id ""LOC127239154""; transcript_id """"; db_x...",LOC127239154,,GeneID:127239154,uncharacterized LOC127239154,Gene,LOC127239154,lncRNA,true
1,NC_060925.1,BestRefSeq,transcript,7506.0,138480.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820...",LOC127239154,NR_182074.1,GeneID:127239154,NaN,ncRNA,LOC127239154,NaN,NaN
2,NC_060925.1,BestRefSeq,exon,138321.0,138480.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820...",LOC127239154,NR_182074.1,GeneID:127239154,NaN,NaN,LOC127239154,NaN,NaN
3,NC_060925.1,BestRefSeq,exon,129906.0,130010.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820...",LOC127239154,NR_182074.1,GeneID:127239154,NaN,NaN,LOC127239154,NaN,NaN
4,NC_060925.1,BestRefSeq,exon,109651.0,109660.0,.,-,.,"gene_id ""LOC127239154""; transcript_id ""NR_1820...",LOC127239154,NR_182074.1,GeneID:127239154,NaN,NaN,LOC127239154,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4298395,NC_060948.1,BestRefSeq,transcript,62449384.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,misc_RNA,DDX11L16,NaN,NaN
4298396,NC_060948.1,BestRefSeq,exon,62451557.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,NaN,DDX11L16,NaN,NaN
4298397,NC_060948.1,BestRefSeq,exon,62451063.0,62451171.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,NaN,DDX11L16,NaN,NaN
4298398,NC_060948.1,BestRefSeq,exon,62449384.0,62450563.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,NaN,DDX11L16,NaN,NaN


In [4]:
par_genes = [
    # PAR1
    "PLCXD1",
    "GTPBP6",
    "PPP2R3B",
    "SHOX",
    "CRLF2",
    "CSF2RA",
    "IL3RA",
    "SLC25A6",
    "ASMTL",
    "P2RY8",
    "AKAP17A",
    "ASMT",
    "DHRSX",
    "ZBED1",
    "CD99",
    "XG",
    # PAR2
    "SPRY3",
    "SYBL1",
    "IL9R",
    "VAMP7"
]

In [5]:
chrYannotation = chm13annotation[chm13annotation["contig"]=="NC_060948.1"]
maleGeneannotation = chrYannotation[~chrYannotation["gene"].isin(par_genes)]
maleGeneannotation

,contig,1,type,start,stop,5,strand,7,info,gene_id,transcript_id,db_xref,description,gbkey,gene,gene_biotype,partial
4238275,NC_060948.1,BestRefSeq,gene,149286.0,151470.0,.,+,.,"gene_id ""LINC00685_1""; transcript_id """"; db_xr...",LINC00685_1,,GeneID:283981,long intergenic non-protein coding RNA 685,Gene,LINC00685,lncRNA,NaN
4238276,NC_060948.1,BestRefSeq,transcript,149286.0,151470.0,.,+,.,"gene_id ""LINC00685_1""; transcript_id ""NR_02723...",LINC00685_1,NR_027232.1_1,GeneID:283981,NaN,ncRNA,LINC00685,NaN,NaN
4238277,NC_060948.1,BestRefSeq,exon,149286.0,151470.0,.,+,.,"gene_id ""LINC00685_1""; transcript_id ""NR_02723...",LINC00685_1,NR_027232.1_1,GeneID:283981,NaN,NaN,LINC00685,NaN,NaN
4238278,NC_060948.1,BestRefSeq,transcript,149286.0,151470.0,.,+,.,"gene_id ""LINC00685_1""; transcript_id ""NR_02723...",LINC00685_1,NR_027231.1_1,GeneID:283981,NaN,ncRNA,LINC00685,NaN,NaN
4238279,NC_060948.1,BestRefSeq,exon,149286.0,149689.0,.,+,.,"gene_id ""LINC00685_1""; transcript_id ""NR_02723...",LINC00685_1,NR_027231.1_1,GeneID:283981,NaN,NaN,LINC00685,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4298394,NC_060948.1,BestRefSeq,gene,62449384.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id """"; db_xre...",DDX11L16_1,,GeneID:727856,DEAD/H-box helicase 11 like 16 (pseudogene),Gene,DDX11L16,transcribed_pseudogene,NaN
4298395,NC_060948.1,BestRefSeq,transcript,62449384.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,misc_RNA,DDX11L16,NaN,NaN
4298396,NC_060948.1,BestRefSeq,exon,62451557.0,62451910.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,NaN,DDX11L16,NaN,NaN
4298397,NC_060948.1,BestRefSeq,exon,62451063.0,62451171.0,.,-,.,"gene_id ""DDX11L16_1""; transcript_id ""NR_110561...",DDX11L16_1,NR_110561.1_1,GeneID:727856,NaN,NaN,DDX11L16,NaN,NaN


In [6]:
exon_maleGeneannotation = maleGeneannotation[maleGeneannotation["type"]=="exon"]

In [7]:
exon_dict = (
    exon_maleGeneannotation.groupby('gene')[['start', 'stop']]
    .apply(lambda x: list(zip(x['start'], x['stop'])))
    .to_dict()
)

In [8]:
def merge_intervals(intervals):
    sorted_intervals = sorted(intervals, key=lambda x: x[0])
    merged = [sorted_intervals[0]]
    
    for start, stop in sorted_intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], stop))
        else:
            merged.append((start, stop))
    
    return merged

exon_dict = (
    exon_maleGeneannotation.groupby('gene')[['start', 'stop']]
    .apply(lambda x: merge_intervals(list(zip(x['start'], x['stop']))))
    .to_dict()
)

In [9]:
set(maleGeneannotation["gene"])

{'LOC124908733',
 'LOC124908982',
 'LOC124908877',
 'LOC124909308',
 'TTTY7B',
 'LOC124908599',
 'LOC124908673',
 'LOC124908678',
 'LOC124908751',
 'OFD1P13Y',
 'TSPY15P',
 'RBMY1D',
 'LOC124908648',
 'LOC124909291',
 'TBL1YP1',
 'LOC124909317',
 'USP9YP30',
 'TRPC6P1',
 'LOC100379236',
 'LOC124908607',
 'ARSFP1',
 'RBMY3AP',
 'LOC124909034',
 'USP9YP35',
 'LOC124908559',
 'TTTY22',
 'LOC124909277',
 'LOC124909279',
 'HSFY6P',
 'SEPTIN14P22',
 'LOC124908913',
 'LOC124909077',
 'LOC124909145',
 'LOC124905649',
 'LOC124908655',
 'LOC124909322',
 'TRIM60P2Y',
 'LOC124905239',
 'LOC124909080',
 'MIR3690',
 'LOC124908936',
 'LOC124908879',
 'TSPY20P',
 'LOC124909334',
 'PSIP1P2',
 'ZNF736P9Y',
 'LOC124909167',
 'LOC124908757',
 'USP9YP25',
 'LOC128966657',
 'LOC124909000',
 'LOC124909225',
 'LOC124908772',
 'LINC00278',
 'CDY8P',
 'ZNF884P',
 'LOC124909335',
 'LOC124908833',
 'TRIM60P9Y',
 'TSPY8',
 'LOC124909131',
 'ELOCP9',
 'LOC124908890',
 'RN7SL818P',
 'LOC124909202',
 'MTCO1P37',
 'LI

# Gencode data

In [10]:
import pysam
import os
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

gene_exon_dict = exon_dict
alignment_dir = "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmedMappedCHM13/"
output_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/gencodeTestis"
os.makedirs(output_dir, exist_ok=True)
target_chrom = "NC_060948.1"

bam_files = [
    os.path.join(alignment_dir, f)
    for f in os.listdir(alignment_dir)
    if f.endswith(('.bam', '.sam'))
]

def process_bam(bam_path):
    # Each worker accumulates reads per gene, returns dict of {gene: {read_name: seq}}
    gene_reads = defaultdict(dict)
    
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for gene, intervals in gene_exon_dict.items():
            for start, stop in intervals:
                for read in bam.fetch(target_chrom, start, stop):
                    if read.is_unmapped or read.query_sequence is None:
                        continue
                    # Use read name as key to deduplicate across intervals
                    gene_reads[gene][read.query_name] = read.query_sequence
    
    return gene_reads

def merge_results(all_results):
    merged = defaultdict(dict)
    for result in all_results:
        for gene, reads in result.items():
            merged[gene].update(reads)
    return merged

with ProcessPoolExecutor() as executor:
    all_results = list(executor.map(process_bam, bam_files))

merged = merge_results(all_results)

for gene, reads in merged.items():
    out_path = os.path.join(output_dir, f"{gene}.fasta")
    with open(out_path, 'w') as f:
        for read_name, seq in reads.items():
            f.write(f">{read_name}\n{seq}\n")

In [11]:
from pathlib import Path
from collections import defaultdict
import re

# Configuration
input_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/gencodeTestis/"
output_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/gencodeTestis/merged/"

# Genes to skip (PAR genes)
# Create output directory
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Group files by base name
base_name_files = defaultdict(list)

for fasta_file in Path(input_dir).glob("*.fa*"):
    # Extract gene name from filename (remove _reads.fa or similar suffixes)
    gene_name = fasta_file.stem.replace("_reads", "")
    
    # Skip if gene name contains any PAR gene
    if any(par_gene in gene_name for par_gene in par_genes):
        continue
    
    # Extract base name
    # For LOC and LINC genes, keep the full name
    # For others, remove trailing digits and letters
    if gene_name.startswith("LOC") or gene_name.startswith("LINC"):
        base_name = gene_name
    else:
        base_name = re.sub(r'[0-9]+[A-Z]*$', '', gene_name)
    
    # Skip if base name is in PAR genes
    if base_name in par_genes:
        continue
    
    # Group files by base name
    base_name_files[base_name].append(fasta_file)

print(f"Found {len(base_name_files)} unique gene base names")
print(f"Processing {sum(len(files) for files in base_name_files.values())} files total\n")

# Merge files for each base name
for base_name, files in base_name_files.items():
    output_file = Path(output_dir) / f"{base_name}_mergedReads.fa"
    
    with open(output_file, 'w') as out:
        for fasta_file in files:
            with open(fasta_file, 'r') as infile:
                out.write(infile.read())
    
    print(f"{base_name}: merged {len(files)} file(s) -> {output_file.name}")

print(f"\nMerging complete! Output in: {output_dir}")

Found 570 unique gene base names
Processing 639 files total

AMELY: merged 1 file(s) -> AMELY_mergedReads.fa
BCORP: merged 1 file(s) -> BCORP_mergedReads.fa
BPY: merged 3 file(s) -> BPY_mergedReads.fa
CDY: merged 4 file(s) -> CDY_mergedReads.fa
CSPG4P: merged 1 file(s) -> CSPG4P_mergedReads.fa
DAZ: merged 4 file(s) -> DAZ_mergedReads.fa
DDX11L: merged 1 file(s) -> DDX11L_mergedReads.fa
DDX: merged 1 file(s) -> DDX_mergedReads.fa
EIF: merged 1 file(s) -> EIF_mergedReads.fa
FAM197Y: merged 6 file(s) -> FAM197Y_mergedReads.fa
FAM: merged 2 file(s) -> FAM_mergedReads.fa
FAM41AY: merged 2 file(s) -> FAM41AY_mergedReads.fa
GOLGA2P: merged 2 file(s) -> GOLGA2P_mergedReads.fa
GYG2P: merged 1 file(s) -> GYG2P_mergedReads.fa
HSFY: merged 2 file(s) -> HSFY_mergedReads.fa
KDM: merged 1 file(s) -> KDM_mergedReads.fa
LINC00102: merged 1 file(s) -> LINC00102_mergedReads.fa
LINC00106: merged 1 file(s) -> LINC00106_mergedReads.fa
LINC00278: merged 1 file(s) -> LINC00278_mergedReads.fa
LINC00279: merged

LOC124908652: merged 1 file(s) -> LOC124908652_mergedReads.fa
LOC124908653: merged 1 file(s) -> LOC124908653_mergedReads.fa
LOC124908654: merged 1 file(s) -> LOC124908654_mergedReads.fa
LOC124908655: merged 1 file(s) -> LOC124908655_mergedReads.fa
LOC124908656: merged 1 file(s) -> LOC124908656_mergedReads.fa
LOC124908657: merged 1 file(s) -> LOC124908657_mergedReads.fa
LOC124908658: merged 1 file(s) -> LOC124908658_mergedReads.fa
LOC124908659: merged 1 file(s) -> LOC124908659_mergedReads.fa
LOC124908660: merged 1 file(s) -> LOC124908660_mergedReads.fa
LOC124908661: merged 1 file(s) -> LOC124908661_mergedReads.fa
LOC124908662: merged 1 file(s) -> LOC124908662_mergedReads.fa
LOC124908663: merged 1 file(s) -> LOC124908663_mergedReads.fa
LOC124908664: merged 1 file(s) -> LOC124908664_mergedReads.fa
LOC124908665: merged 1 file(s) -> LOC124908665_mergedReads.fa
LOC124908671: merged 1 file(s) -> LOC124908671_mergedReads.fa
LOC124908672: merged 1 file(s) -> LOC124908672_mergedReads.fa
LOC12490

LOC124908966: merged 1 file(s) -> LOC124908966_mergedReads.fa
LOC124908968: merged 1 file(s) -> LOC124908968_mergedReads.fa
LOC124908969: merged 1 file(s) -> LOC124908969_mergedReads.fa
LOC124908973: merged 1 file(s) -> LOC124908973_mergedReads.fa
LOC124908974: merged 1 file(s) -> LOC124908974_mergedReads.fa
LOC124908978: merged 1 file(s) -> LOC124908978_mergedReads.fa
LOC124908979: merged 1 file(s) -> LOC124908979_mergedReads.fa
LOC124908980: merged 1 file(s) -> LOC124908980_mergedReads.fa
LOC124908981: merged 1 file(s) -> LOC124908981_mergedReads.fa
LOC124908983: merged 1 file(s) -> LOC124908983_mergedReads.fa
LOC124908984: merged 1 file(s) -> LOC124908984_mergedReads.fa
LOC124908985: merged 1 file(s) -> LOC124908985_mergedReads.fa
LOC124908986: merged 1 file(s) -> LOC124908986_mergedReads.fa
LOC124908987: merged 1 file(s) -> LOC124908987_mergedReads.fa
LOC124908988: merged 1 file(s) -> LOC124908988_mergedReads.fa
LOC124908991: merged 1 file(s) -> LOC124908991_mergedReads.fa
LOC12490

LOC124909210: merged 1 file(s) -> LOC124909210_mergedReads.fa
LOC124909213: merged 1 file(s) -> LOC124909213_mergedReads.fa
LOC124909214: merged 1 file(s) -> LOC124909214_mergedReads.fa
LOC124909215: merged 1 file(s) -> LOC124909215_mergedReads.fa
LOC124909218: merged 1 file(s) -> LOC124909218_mergedReads.fa
LOC124909225: merged 1 file(s) -> LOC124909225_mergedReads.fa
LOC124909226: merged 1 file(s) -> LOC124909226_mergedReads.fa
LOC124909227: merged 1 file(s) -> LOC124909227_mergedReads.fa
LOC124909234: merged 1 file(s) -> LOC124909234_mergedReads.fa
LOC124909235: merged 1 file(s) -> LOC124909235_mergedReads.fa
LOC124909237: merged 1 file(s) -> LOC124909237_mergedReads.fa
LOC124909238: merged 1 file(s) -> LOC124909238_mergedReads.fa
LOC124909242: merged 1 file(s) -> LOC124909242_mergedReads.fa
LOC124909245: merged 1 file(s) -> LOC124909245_mergedReads.fa
LOC124909246: merged 1 file(s) -> LOC124909246_mergedReads.fa
LOC124909247: merged 1 file(s) -> LOC124909247_mergedReads.fa
LOC12490

In [3]:
import gzip
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
FASTA_FILE = "/project/mkonkel/tangeno/users/giannim/chrY/RBMY1B_good_cleanish.fasta"   # path to your aligned FASTA file
FASTQ_FILE = "/project/mkonkel/tangeno/users/giannim/chrY/extraSamples/trimmed/SRR31360662_trimmed.fastq"     # path to your FASTQ file (plain or .gz)
# ──────────────────────────────────────────────────────────────────────────────


def open_file(path):
    """Open plain or gzip-compressed files transparently."""
    p = Path(path)
    if p.suffix == ".gz":
        return gzip.open(p, "rt")
    return open(p, "r")


def reverse_complement(seq):
    comp = str.maketrans("ACGTacgtNn", "TGCAtgcaNn")
    return seq.translate(comp)[::-1]


def parse_fasta(path):
    """
    Load aligned FASTA into memory as {read_id: ungapped_sequence}.
    Also builds a reverse-complement lookup so we never recompute it.
    """
    records = {}
    current_id = None
    seq_parts = []

    with open_file(path) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    seq = "".join(seq_parts).replace("-", "").upper()
                    records[current_id] = seq
                current_id = line[1:].split()[0]
                seq_parts = []
            else:
                seq_parts.append(line)

    if current_id is not None:
        seq = "".join(seq_parts).replace("-", "").upper()
        records[current_id] = seq

    return records


def stream_fastq(path):
    """
    Generator: yield (read_id, sequence) for each record in a FASTQ file.
    Never holds more than one record in memory at a time.
    """
    with open_file(path) as fh:
        while True:
            header = fh.readline()
            if not header:
                return                         # EOF
            seq    = fh.readline().rstrip().upper()
            fh.readline()                      # '+' line  (discard)
            fh.readline()                      # quality   (discard)
            read_id = header.rstrip().lstrip("@").split()[0]
            yield read_id, seq


# ── Step 1: load FASTA (small) into memory ────────────────────────────────────
print("Parsing FASTA ...")
fasta_records = parse_fasta(FASTA_FILE)
print(f"  {len(fasta_records):,} sequences loaded")

# Pre-compute reverse complements so we don't redo them per FASTQ read
fasta_rc = {rid: reverse_complement(seq) for rid, seq in fasta_records.items()}

# Track which IDs we still need to find in the FASTQ
remaining = set(fasta_records.keys())

mismatched     = []  # seq differs even after RC check
extra_in_fastq = 0   # reads in FASTQ not in FASTA (counted, not stored)
i              = 0

# ── Step 2: single streaming pass over the FASTQ (never fully loaded) ─────────
print(f"\nStreaming FASTQ (this may take a while for large files) ...")

for i, (read_id, fastq_seq) in enumerate(stream_fastq(FASTQ_FILE), 1):
    if i % 1_000_000 == 0:
        print(f"  {i:,} reads processed, {len(remaining):,} FASTA reads still to find ...")

    if read_id not in fasta_records:
        extra_in_fastq += 1
        continue

    # We found this read — remove from the "still looking" set
    remaining.discard(read_id)

    fasta_seq = fasta_records[read_id]

    if fastq_seq == fasta_seq:
        continue                               # forward match

    if fastq_seq == fasta_rc[read_id]:
        continue                               # reverse-complement match

    mismatched.append({
        "read_id":   read_id,
        "fasta_seq": fasta_seq,
        "rc_seq":    fasta_rc[read_id],
        "fastq_seq": fastq_seq,
    })

# Anything left in `remaining` was never seen in the FASTQ
missing_from_fastq = sorted(remaining)

# ── Report ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"FASTQ reads scanned       : {i:,}")
print(f"FASTA reads checked       : {len(fasta_records):,}")
print(f"Missing from FASTQ        : {len(missing_from_fastq):,}")
print(f"Extra in FASTQ (not FASTA): {extra_in_fastq:,}")
print(f"Mismatched reads          : {len(mismatched):,}")
print("=" * 60)

if missing_from_fastq:
    print("\nRead IDs in FASTA but never found in FASTQ:")
    for rid in missing_from_fastq:
        print(f"  {rid}")

if mismatched:
    print("\nRead IDs whose sequences do not match (forward OR reverse complement):")
    for m in mismatched:
        print(f"\n  ID       : {m['read_id']}")
        print(f"  FASTA    : {m['fasta_seq'][:80]}{'...' if len(m['fasta_seq']) > 80 else ''}")
        print(f"  FASTA RC : {m['rc_seq'][:80]}{'...' if len(m['rc_seq']) > 80 else ''}")
        print(f"  FASTQ    : {m['fastq_seq'][:80]}{'...' if len(m['fastq_seq']) > 80 else ''}")
else:
    print("\nAll FASTA sequences match their FASTQ counterparts.")

# Flat list of mismatched IDs for downstream use
mismatched_ids = [m["read_id"] for m in mismatched]

Parsing FASTA ...
  525 sequences loaded

Streaming FASTQ (this may take a while for large files) ...
  1,000,000 reads processed, 513 FASTA reads still to find ...
  2,000,000 reads processed, 502 FASTA reads still to find ...
  3,000,000 reads processed, 496 FASTA reads still to find ...
  4,000,000 reads processed, 490 FASTA reads still to find ...
  5,000,000 reads processed, 474 FASTA reads still to find ...
  6,000,000 reads processed, 467 FASTA reads still to find ...
  7,000,000 reads processed, 459 FASTA reads still to find ...
  8,000,000 reads processed, 450 FASTA reads still to find ...
  9,000,000 reads processed, 438 FASTA reads still to find ...
  10,000,000 reads processed, 434 FASTA reads still to find ...
  11,000,000 reads processed, 425 FASTA reads still to find ...
  12,000,000 reads processed, 415 FASTA reads still to find ...
  13,000,000 reads processed, 412 FASTA reads still to find ...
  14,000,000 reads processed, 409 FASTA reads still to find ...
  15,000,00

In [4]:
import gzip
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
FASTA_FILE = "/project/mkonkel/tangeno/users/giannim/chrY/RBMY1B_good_cleanish.fasta"   # path to your aligned FASTA file
FASTQ_FILE = "/project/mkonkel/tangeno/users/giannim/testis/01_originalFiles/SRR12544672.fastq"     # path to your FASTQ file (plain or .gz)
# ──────────────────────────────────────────────────────────────────────────────


def open_file(path):
    """Open plain or gzip-compressed files transparently."""
    p = Path(path)
    if p.suffix == ".gz":
        return gzip.open(p, "rt")
    return open(p, "r")


def reverse_complement(seq):
    comp = str.maketrans("ACGTacgtNn", "TGCAtgcaNn")
    return seq.translate(comp)[::-1]


def parse_fasta(path):
    """
    Load aligned FASTA into memory as {read_id: ungapped_sequence}.
    Also builds a reverse-complement lookup so we never recompute it.
    """
    records = {}
    current_id = None
    seq_parts = []

    with open_file(path) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    seq = "".join(seq_parts).replace("-", "").upper()
                    records[current_id] = seq
                current_id = line[1:].split()[0]
                seq_parts = []
            else:
                seq_parts.append(line)

    if current_id is not None:
        seq = "".join(seq_parts).replace("-", "").upper()
        records[current_id] = seq

    return records


def stream_fastq(path):
    """
    Generator: yield (read_id, sequence) for each record in a FASTQ file.
    Never holds more than one record in memory at a time.
    """
    with open_file(path) as fh:
        while True:
            header = fh.readline()
            if not header:
                return                         # EOF
            seq    = fh.readline().rstrip().upper()
            fh.readline()                      # '+' line  (discard)
            fh.readline()                      # quality   (discard)
            read_id = header.rstrip().lstrip("@").split()[0]
            yield read_id, seq


# ── Step 1: load FASTA (small) into memory ────────────────────────────────────
print("Parsing FASTA ...")
fasta_records = parse_fasta(FASTA_FILE)
print(f"  {len(fasta_records):,} sequences loaded")

# Pre-compute reverse complements so we don't redo them per FASTQ read
fasta_rc = {rid: reverse_complement(seq) for rid, seq in fasta_records.items()}

# Track which IDs we still need to find in the FASTQ
remaining = set(fasta_records.keys())

mismatched     = []  # seq differs even after RC check
extra_in_fastq = 0   # reads in FASTQ not in FASTA (counted, not stored)
i              = 0

# ── Step 2: single streaming pass over the FASTQ (never fully loaded) ─────────
print(f"\nStreaming FASTQ (this may take a while for large files) ...")

for i, (read_id, fastq_seq) in enumerate(stream_fastq(FASTQ_FILE), 1):
    if i % 1_000_000 == 0:
        print(f"  {i:,} reads processed, {len(remaining):,} FASTA reads still to find ...")

    if read_id not in fasta_records:
        extra_in_fastq += 1
        continue

    # We found this read — remove from the "still looking" set
    remaining.discard(read_id)

    fasta_seq = fasta_records[read_id]

    if fastq_seq == fasta_seq:
        continue                               # forward match

    if fastq_seq == fasta_rc[read_id]:
        continue                               # reverse-complement match

    mismatched.append({
        "read_id":   read_id,
        "fasta_seq": fasta_seq,
        "rc_seq":    fasta_rc[read_id],
        "fastq_seq": fastq_seq,
    })

# Anything left in `remaining` was never seen in the FASTQ
missing_from_fastq = sorted(remaining)

# ── Report ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"FASTQ reads scanned       : {i:,}")
print(f"FASTA reads checked       : {len(fasta_records):,}")
print(f"Missing from FASTQ        : {len(missing_from_fastq):,}")
print(f"Extra in FASTQ (not FASTA): {extra_in_fastq:,}")
print(f"Mismatched reads          : {len(mismatched):,}")
print("=" * 60)

if missing_from_fastq:
    print("\nRead IDs in FASTA but never found in FASTQ:")
    for rid in missing_from_fastq:
        print(f"  {rid}")

if mismatched:
    print("\nRead IDs whose sequences do not match (forward OR reverse complement):")
    for m in mismatched:
        print(f"\n  ID       : {m['read_id']}")
        print(f"  FASTA    : {m['fasta_seq'][:80]}{'...' if len(m['fasta_seq']) > 80 else ''}")
        print(f"  FASTA RC : {m['rc_seq'][:80]}{'...' if len(m['rc_seq']) > 80 else ''}")
        print(f"  FASTQ    : {m['fastq_seq'][:80]}{'...' if len(m['fastq_seq']) > 80 else ''}")
else:
    print("\nAll FASTA sequences match their FASTQ counterparts.")

# Flat list of mismatched IDs for downstream use
mismatched_ids = [m["read_id"] for m in mismatched]

Parsing FASTA ...
  525 sequences loaded

Streaming FASTQ (this may take a while for large files) ...

FASTQ reads scanned       : 606,488
FASTA reads checked       : 525
Missing from FASTQ        : 472
Extra in FASTQ (not FASTA): 606,435
Mismatched reads          : 12

Read IDs in FASTA but never found in FASTQ:
  200080_RBMY1B
  ENST00000382659.7
  ENST00000382680.5
  ENST00000382707.6
  ENST00000383020.7
  HC02666_RBMY_10_consensus
  HC02666_RBMY_11_consensus
  HC02666_RBMY_4_consensus
  HC19384_RBMY_10
  HC19384_RBMY_11
  HC19384_RBMY_12
  HC19384_RBMY_14
  HC19384_RBMY_6
  HC19384_RBMY_7
  HC19384_RBMY_8
  HC19384_RBMY_9
  HG00096_RBMY1B
  HG00512_RBMY_7_consensus
  HG00673_RBMY_9_consensus_1SNV
  HG01109_RBMY_11_consensus
  HG01109_RBMY_12_consensus
  HG01890_RBMY_10
  HG01890_RBMY_11
  HG01890_RBMY_12
  HG01890_RBMY_13
  HG01890_RBMY_14
  HG01890_RBMY_16
  HG01890_RBMY_16_consensus
  HG01890_RBMY_6_consensus
  HG01890_RBMY_7
  HG01890_RBMY_7_consensus
  HG01890_RBMY_8_consensus


In [5]:
import gzip
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
FASTA_FILE = "/project/mkonkel/tangeno/users/giannim/chrY/RBMY1B_good_cleanish.fasta"   # path to your aligned FASTA file
FASTQ_FILE = "/project/mkonkel/tangeno/users/giannim/testis/01_originalFiles/SRR12544673.fastq"     # path to your FASTQ file (plain or .gz)
# ──────────────────────────────────────────────────────────────────────────────


def open_file(path):
    """Open plain or gzip-compressed files transparently."""
    p = Path(path)
    if p.suffix == ".gz":
        return gzip.open(p, "rt")
    return open(p, "r")


def reverse_complement(seq):
    comp = str.maketrans("ACGTacgtNn", "TGCAtgcaNn")
    return seq.translate(comp)[::-1]


def parse_fasta(path):
    """
    Load aligned FASTA into memory as {read_id: ungapped_sequence}.
    Also builds a reverse-complement lookup so we never recompute it.
    """
    records = {}
    current_id = None
    seq_parts = []

    with open_file(path) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    seq = "".join(seq_parts).replace("-", "").upper()
                    records[current_id] = seq
                current_id = line[1:].split()[0]
                seq_parts = []
            else:
                seq_parts.append(line)

    if current_id is not None:
        seq = "".join(seq_parts).replace("-", "").upper()
        records[current_id] = seq

    return records


def stream_fastq(path):
    """
    Generator: yield (read_id, sequence) for each record in a FASTQ file.
    Never holds more than one record in memory at a time.
    """
    with open_file(path) as fh:
        while True:
            header = fh.readline()
            if not header:
                return                         # EOF
            seq    = fh.readline().rstrip().upper()
            fh.readline()                      # '+' line  (discard)
            fh.readline()                      # quality   (discard)
            read_id = header.rstrip().lstrip("@").split()[0]
            yield read_id, seq


# ── Step 1: load FASTA (small) into memory ────────────────────────────────────
print("Parsing FASTA ...")
fasta_records = parse_fasta(FASTA_FILE)
print(f"  {len(fasta_records):,} sequences loaded")

# Pre-compute reverse complements so we don't redo them per FASTQ read
fasta_rc = {rid: reverse_complement(seq) for rid, seq in fasta_records.items()}

# Track which IDs we still need to find in the FASTQ
remaining = set(fasta_records.keys())

mismatched     = []  # seq differs even after RC check
extra_in_fastq = 0   # reads in FASTQ not in FASTA (counted, not stored)
i              = 0

# ── Step 2: single streaming pass over the FASTQ (never fully loaded) ─────────
print(f"\nStreaming FASTQ (this may take a while for large files) ...")

for i, (read_id, fastq_seq) in enumerate(stream_fastq(FASTQ_FILE), 1):
    if i % 1_000_000 == 0:
        print(f"  {i:,} reads processed, {len(remaining):,} FASTA reads still to find ...")

    if read_id not in fasta_records:
        extra_in_fastq += 1
        continue

    # We found this read — remove from the "still looking" set
    remaining.discard(read_id)

    fasta_seq = fasta_records[read_id]

    if fastq_seq == fasta_seq:
        continue                               # forward match

    if fastq_seq == fasta_rc[read_id]:
        continue                               # reverse-complement match

    mismatched.append({
        "read_id":   read_id,
        "fasta_seq": fasta_seq,
        "rc_seq":    fasta_rc[read_id],
        "fastq_seq": fastq_seq,
    })

# Anything left in `remaining` was never seen in the FASTQ
missing_from_fastq = sorted(remaining)

# ── Report ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"FASTQ reads scanned       : {i:,}")
print(f"FASTA reads checked       : {len(fasta_records):,}")
print(f"Missing from FASTQ        : {len(missing_from_fastq):,}")
print(f"Extra in FASTQ (not FASTA): {extra_in_fastq:,}")
print(f"Mismatched reads          : {len(mismatched):,}")
print("=" * 60)

if missing_from_fastq:
    print("\nRead IDs in FASTA but never found in FASTQ:")
    for rid in missing_from_fastq:
        print(f"  {rid}")

if mismatched:
    print("\nRead IDs whose sequences do not match (forward OR reverse complement):")
    for m in mismatched:
        print(f"\n  ID       : {m['read_id']}")
        print(f"  FASTA    : {m['fasta_seq'][:80]}{'...' if len(m['fasta_seq']) > 80 else ''}")
        print(f"  FASTA RC : {m['rc_seq'][:80]}{'...' if len(m['rc_seq']) > 80 else ''}")
        print(f"  FASTQ    : {m['fastq_seq'][:80]}{'...' if len(m['fastq_seq']) > 80 else ''}")
else:
    print("\nAll FASTA sequences match their FASTQ counterparts.")

# Flat list of mismatched IDs for downstream use
mismatched_ids = [m["read_id"] for m in mismatched]

Parsing FASTA ...
  525 sequences loaded

Streaming FASTQ (this may take a while for large files) ...

FASTQ reads scanned       : 551,565
FASTA reads checked       : 525
Missing from FASTQ        : 509
Extra in FASTQ (not FASTA): 551,549
Mismatched reads          : 4

Read IDs in FASTA but never found in FASTQ:
  200080_RBMY1B
  ENST00000382659.7
  ENST00000382680.5
  ENST00000382707.6
  ENST00000383020.7
  HC02666_RBMY_10_consensus
  HC02666_RBMY_11_consensus
  HC02666_RBMY_4_consensus
  HC19384_RBMY_10
  HC19384_RBMY_11
  HC19384_RBMY_12
  HC19384_RBMY_14
  HC19384_RBMY_6
  HC19384_RBMY_7
  HC19384_RBMY_8
  HC19384_RBMY_9
  HG00096_RBMY1B
  HG00512_RBMY_7_consensus
  HG00673_RBMY_9_consensus_1SNV
  HG01109_RBMY_11_consensus
  HG01109_RBMY_12_consensus
  HG01890_RBMY_10
  HG01890_RBMY_11
  HG01890_RBMY_12
  HG01890_RBMY_13
  HG01890_RBMY_14
  HG01890_RBMY_16
  HG01890_RBMY_16_consensus
  HG01890_RBMY_6_consensus
  HG01890_RBMY_7
  HG01890_RBMY_7_consensus
  HG01890_RBMY_8_consensus
 

# Great Apes

In [12]:
import pysam
import os
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

gene_exon_dict = exon_dict
alignment_dir = "/project/mkonkel/tangeno/users/giannim/apeTestis/00_mapReadsCHM13/"
output_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis"
os.makedirs(output_dir, exist_ok=True)
target_chrom = "NC_060948.1"

bam_files = [
    os.path.join(alignment_dir, f)
    for f in os.listdir(alignment_dir)
    if f.endswith(('.bam', '.sam'))
]

def process_bam(bam_path):
    gene_reads = defaultdict(dict)
    
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for gene, intervals in gene_exon_dict.items():
            for start, stop in intervals:
                for read in bam.fetch(target_chrom, start, stop):
                    if read.is_unmapped or read.query_sequence is None:
                        continue
                    gene_reads[gene][read.query_name] = read.query_sequence
    
    return bam_path, gene_reads

def merge_results(all_results):
    merged = defaultdict(dict)
    for _, gene_reads in all_results:
        for gene, reads in gene_reads.items():
            merged[gene].update(reads)
    return merged

with ProcessPoolExecutor() as executor:
    all_results = list(executor.map(process_bam, bam_files))

# Write per-sample directories
for bam_path, gene_reads in all_results:
    sample_name = os.path.splitext(os.path.basename(bam_path))[0]
    sample_dir = os.path.join(output_dir, sample_name)
    os.makedirs(sample_dir, exist_ok=True)

    for gene, reads in gene_reads.items():
        if reads:
            out_path = os.path.join(sample_dir, f"{gene}.fasta")
            with open(out_path, 'w') as f:
                for read_name, seq in reads.items():
                    f.write(f">{read_name}\n{seq}\n")

# Write merged output across all samples
merged = merge_results(all_results)
for gene, reads in merged.items():
    if reads:
        out_path = os.path.join(output_dir, f"{gene}.fasta")
        with open(out_path, 'w') as f:
            for read_name, seq in reads.items():
                f.write(f">{read_name}\n{seq}\n")

In [13]:
from pathlib import Path
from collections import defaultdict
import re

# Configuration
input_dir = Path("/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis")

def merge_fasta_dir(sample_dir, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    base_name_files = defaultdict(list)
    for fasta_file in sample_dir.glob("*.fa*"):
        gene_name = fasta_file.stem.replace("_reads", "")
        
        if any(par_gene in gene_name for par_gene in par_genes):
            continue
        
        if gene_name.startswith("LOC") or gene_name.startswith("LINC"):
            base_name = gene_name
        else:
            base_name = re.sub(r'[0-9]+[A-Z]*$', '', gene_name)
        
        if base_name in par_genes:
            continue
        
        base_name_files[base_name].append(fasta_file)
    
    for base_name, files in base_name_files.items():
        output_file = output_dir / f"{base_name}_mergedReads.fa"
        with open(output_file, 'w') as out:
            for fasta_file in files:
                with open(fasta_file, 'r') as infile:
                    out.write(infile.read())
    
    return len(base_name_files)

# Process top-level fasta files
top_merged_dir = input_dir / "merged"
n = merge_fasta_dir(input_dir, top_merged_dir)
print(f"Top-level: merged {n} gene base names -> {top_merged_dir}")

# Process each sample subdirectory
for sample_dir in sorted(input_dir.iterdir()):
    if not sample_dir.is_dir() or sample_dir.name == "merged":
        continue
    
    sample_merged_dir = sample_dir / "merged"
    n = merge_fasta_dir(sample_dir, sample_merged_dir)
    print(f"{sample_dir.name}: merged {n} gene base names -> {sample_merged_dir}")

print("\nMerging complete!")

Top-level: merged 0 gene base names -> /project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis/merged
SRR22838391_subreads_trimmed: merged 57 gene base names -> /project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis/SRR22838391_subreads_trimmed/merged
SRR22838392_subreads_trimmed: merged 53 gene base names -> /project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis/SRR22838392_subreads_trimmed/merged
SRR22838393_subreads_trimmed: merged 54 gene base names -> /project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis/SRR22838393_subreads_trimmed/merged
SRR22838394_subreads_trimmed: merged 60 gene base names -> /project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis/SRR22838394_subreads_trimmed/merged
SRR22838395_subreads_trimmed: merged 58 gene base names -> /project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/greatApeTestis/SRR22838395_subreads_trimmed/merged
SRR22838396_subreads_trimmed:

# Testis

In [9]:
import pysam
import os
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

gene_exon_dict = exon_dict
alignment_dir = "/project/mkonkel/tangeno/users/giannim/testis/00_readsMappedCHM13"
output_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/testis"
os.makedirs(output_dir, exist_ok=True)
target_chrom = "NC_060948.1"

bam_files = [
    os.path.join(alignment_dir, f)
    for f in os.listdir(alignment_dir)
    if f.endswith(('.bam', '.sam'))
]

def process_bam(bam_path):
    # Each worker accumulates reads per gene, returns dict of {gene: {read_name: seq}}
    gene_reads = defaultdict(dict)
    
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for gene, intervals in gene_exon_dict.items():
            for start, stop in intervals:
                for read in bam.fetch(target_chrom, start, stop):
                    if read.is_unmapped or read.query_sequence is None:
                        continue
                    # Use read name as key to deduplicate across intervals
                    gene_reads[gene][read.query_name] = read.query_sequence
    
    return gene_reads

def merge_results(all_results):
    merged = defaultdict(dict)
    for result in all_results:
        for gene, reads in result.items():
            merged[gene].update(reads)
    return merged

with ProcessPoolExecutor() as executor:
    all_results = list(executor.map(process_bam, bam_files))

merged = merge_results(all_results)

for gene, reads in merged.items():
    out_path = os.path.join(output_dir, f"{gene}.fasta")
    with open(out_path, 'w') as f:
        for read_name, seq in reads.items():
            f.write(f">{read_name}\n{seq}\n")

In [10]:
from pathlib import Path
from collections import defaultdict
import re

# Configuration
input_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/testis/"
output_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/testis/merged/"

# Genes to skip (PAR genes)
# Create output directory
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Group files by base name
base_name_files = defaultdict(list)

for fasta_file in Path(input_dir).glob("*.fa*"):
    # Extract gene name from filename (remove _reads.fa or similar suffixes)
    gene_name = fasta_file.stem.replace("_reads", "")
    
    # Skip if gene name contains any PAR gene
    if any(par_gene in gene_name for par_gene in par_genes):
        continue
    
    # Extract base name
    # For LOC and LINC genes, keep the full name
    # For others, remove trailing digits and letters
    if gene_name.startswith("LOC") or gene_name.startswith("LINC"):
        base_name = gene_name
    else:
        base_name = re.sub(r'[0-9]+[A-Z]*$', '', gene_name)
    
    # Skip if base name is in PAR genes
    if base_name in par_genes:
        continue
    
    # Group files by base name
    base_name_files[base_name].append(fasta_file)

print(f"Found {len(base_name_files)} unique gene base names")
print(f"Processing {sum(len(files) for files in base_name_files.values())} files total\n")

# Merge files for each base name
for base_name, files in base_name_files.items():
    output_file = Path(output_dir) / f"{base_name}_mergedReads.fa"
    
    with open(output_file, 'w') as out:
        for fasta_file in files:
            with open(fasta_file, 'r') as infile:
                out.write(infile.read())
    
    print(f"{base_name}: merged {len(files)} file(s) -> {output_file.name}")

print(f"\nMerging complete! Output in: {output_dir}")

Found 111 unique gene base names
Processing 155 files total

AMELY: merged 1 file(s) -> AMELY_mergedReads.fa
BCORP: merged 1 file(s) -> BCORP_mergedReads.fa
BPY: merged 3 file(s) -> BPY_mergedReads.fa
CDY: merged 4 file(s) -> CDY_mergedReads.fa
DAZ: merged 4 file(s) -> DAZ_mergedReads.fa
DDX: merged 1 file(s) -> DDX_mergedReads.fa
EIF: merged 1 file(s) -> EIF_mergedReads.fa
FAM: merged 2 file(s) -> FAM_mergedReads.fa
FAM41AY: merged 1 file(s) -> FAM41AY_mergedReads.fa
HSFY: merged 2 file(s) -> HSFY_mergedReads.fa
KDM: merged 1 file(s) -> KDM_mergedReads.fa
LINC00102: merged 1 file(s) -> LINC00102_mergedReads.fa
LINC00106: merged 1 file(s) -> LINC00106_mergedReads.fa
LINC00279: merged 1 file(s) -> LINC00279_mergedReads.fa
LOC101928032: merged 1 file(s) -> LOC101928032_mergedReads.fa
LOC101929148: merged 1 file(s) -> LOC101929148_mergedReads.fa
LOC102725532: merged 1 file(s) -> LOC102725532_mergedReads.fa
LOC105377217: merged 1 file(s) -> LOC105377217_mergedReads.fa
LOC105377220: merged 

# Heart and Cerebellum

In [19]:
import pysam
import os
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

gene_exon_dict = exon_dict
alignment_dir = "/project/mkonkel/tangeno/users/giannim/revio/extra/HeartCereb/01_readsMapped/cereb"
output_dir = "/project/mkonkel/tangeno/users/giannim/chrY/geneFamilyReads/cerebellum"
os.makedirs(output_dir, exist_ok=True)
target_chrom = "chrY"

bam_files = [
    os.path.join(alignment_dir, f)
    for f in os.listdir(alignment_dir)
    if f.endswith(('.bam', '.sam'))
]

def process_bam(bam_path):
    # Each worker accumulates reads per gene, returns dict of {gene: {read_name: seq}}
    gene_reads = defaultdict(dict)
    
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for gene, intervals in gene_exon_dict.items():
            for start, stop in intervals:
                for read in bam.fetch(target_chrom, start, stop):
                    if read.is_unmapped or read.query_sequence is None:
                        continue
                    # Use read name as key to deduplicate across intervals
                    gene_reads[gene][read.query_name] = read.query_sequence
    
    return gene_reads

def merge_results(all_results):
    merged = defaultdict(dict)
    for result in all_results:
        for gene, reads in result.items():
            merged[gene].update(reads)
    return merged

with ProcessPoolExecutor() as executor:
    all_results = list(executor.map(process_bam, bam_files))

merged = merge_results(all_results)

for gene, reads in merged.items():
    out_path = os.path.join(output_dir, f"{gene}.fasta")
    with open(out_path, 'w') as f:
        for read_name, seq in reads.items():
            f.write(f">{read_name}\n{seq}\n")

ValueError: invalid contig `NC_060948.1`

In [21]:
import glob
files = glob.glob("/project/mkonkel/tangeno/forMark/chrY/*.fa")
len(files)

106

In [22]:
files = glob.glob("/project/mkonkel/tangeno/forMark/chrY/*/*.fa")
len(files)

37

# UHRR